# Lab 05 - Retrieval Evaluation: Precision, Recall, MAP, MRR

**Knowledge base:** `knowledge_base/03_retrieval/06_retrieval_evaluation.md`

**Concepts:** Ground truth · Precision@K · Recall@K · MAP@K · MRR · Precision-recall tradeoff

You cannot improve what you cannot measure. This lab gives you the tools to
quantitatively evaluate any retriever.

---

## Setup

This lab uses the **20 Newsgroups** dataset - a classic labelled text dataset.
Pre-computed embeddings are in `embeddings.joblib` (copy from the dataset folder).
If the file is missing, the cell will compute them (takes ~10 min on CPU).

In [1]:
import numpy as np
import pandas as pd
import joblib
import os
from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer

# Load the dataset - downloads ~14 MB on first run
print("Loading 20 Newsgroups dataset...")
newsgroups = fetch_20newsgroups(
    subset="train", shuffle=True, random_state=42,
    data_home="./dataset"   # stored locally - no re-download on next run
)

df = pd.DataFrame({"text": newsgroups.data, "category": newsgroups.target})
print(f"✅ Dataset loaded: {df.shape[0]} documents, {len(newsgroups.target_names)} categories")
print(f"   Categories: {newsgroups.target_names[:5]}... (and {len(newsgroups.target_names)-5} more)")

Loading 20 Newsgroups dataset...
✅ Dataset loaded: 11314 documents, 20 categories
   Categories: ['alt.atheism', 'comp.graphics', 'comp.os.ms-windows.misc', 'comp.sys.ibm.pc.hardware', 'comp.sys.mac.hardware']... (and 15 more)


In [4]:
MODEL_NAME = "BAAI/bge-base-en-v1.5"

# Load pre-computed embeddings (fast) or compute them (slow)
EMBED_FILE = "embeddings.joblib"
if os.path.exists(EMBED_FILE):
    embedding_vectors = joblib.load(EMBED_FILE)
    print(f"✅ Loaded pre-computed embeddings: {len(embedding_vectors)} vectors")
else:
    print("No cached embeddings found. Computing from scratch (~10 min on CPU)...")
    model_em = SentenceTransformer(MODEL_NAME)
    texts    = [t.strip() for t in df["text"].tolist()]
    embedding_vectors = model_em.encode(texts, show_progress_bar=True, batch_size=32)
    joblib.dump(embedding_vectors, EMBED_FILE)
    print(f"✅ Computed and cached {len(embedding_vectors)} embeddings")

print(f"   Shape: {np.array(embedding_vectors).shape}")

No cached embeddings found. Computing from scratch (~10 min on CPU)...


OSError: The paging file is too small for this operation to complete. (os error 1455)

In [ ]:
# Load model for query embedding (needed even with cached doc embeddings)
print("Loading embedding model for query encoding...")
model = SentenceTransformer(MODEL_NAME)
print("✅ Model ready")

def cosine_similarity(v1, matrix):
    v1 = np.array(v1, dtype=np.float32).ravel()
    M  = np.atleast_2d(np.array(matrix, dtype=np.float32))
    v1_norm = np.linalg.norm(v1)
    M_norms = np.linalg.norm(M, axis=1)
    denom   = v1_norm * M_norms
    return (M @ v1 / np.where(denom == 0, 1.0, denom)).tolist()


# Pre-normalize embedding matrix for fast cosine (vectorized)
E = np.array(embedding_vectors, dtype=np.float32)
E_norms = np.linalg.norm(E, axis=1, keepdims=True)
E_norm  = E / np.where(E_norms == 0, 1.0, E_norms)
print(f"✅ Normalized embedding matrix ready: {E_norm.shape}")

---
## 1 - The retrieval function

A basic semantic retriever using pre-computed embeddings.
We retrieve top-K documents for each query and then evaluate the results.

In [ ]:
def retrieve_documents(query: str, top_k: int = 5) -> list:
    '''
    Retrieve top-k documents for a query using cosine similarity over pre-embedded corpus.
    Returns list of (category_name, doc_text_preview) tuples.
    '''
    q_emb = model.encode(query.strip(), convert_to_tensor=False).astype(np.float32)
    q_emb = q_emb / np.linalg.norm(q_emb)   # normalize for fast dot product
    scores = (E_norm @ q_emb).tolist()

    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    results = []
    for idx in top_indices:
        cat  = newsgroups.target_names[df.iloc[idx]["category"]]
        text = df.iloc[idx]["text"][:150].replace("\n", " ").strip()
        results.append((cat, text))
    return results


# Example query
example_query = "advancements in space exploration technology"
print(f"Query: '{example_query}'\n")
for cat, text in retrieve_documents(example_query, top_k=3):
    print(f"  [{cat}] {text}...")

---
## 2 - Precision@K

Precision@K = what fraction of the top-K retrieved documents are relevant?

$$\text{Precision@K} = \frac{\text{relevant documents in top-K}}{K}$$

Measures: **trustworthiness** - can you trust that retrieved results are relevant?

In [ ]:
def precision_at_k(relevant_count: int, k: int) -> float:
    '''Fraction of top-K retrieved documents that are relevant.'''
    if k == 0:
        return 0.0
    if relevant_count < 0 or k < 0:
        raise ValueError("Inputs must be non-negative")
    return relevant_count / k


# Manual example
print("Precision@K examples:")
print(f"  3 relevant out of 5 retrieved: {precision_at_k(3, 5):.2f}")
print(f"  5 relevant out of 5 retrieved: {precision_at_k(5, 5):.2f}  (perfect)")
print(f"  0 relevant out of 5 retrieved: {precision_at_k(0, 5):.2f}  (worst)")

---
## 3 - Recall@K

Recall@K = what fraction of ALL relevant documents in the corpus did we find in top-K?

$$\text{Recall@K} = \frac{\text{relevant documents in top-K}}{\text{total relevant documents in corpus}}$$

Measures: **comprehensiveness** - are we finding everything that matters?

In [ ]:
def recall_at_k(relevant_count: int, total_relevant: int) -> float:
    '''Fraction of all relevant documents in corpus that appear in top-K.'''
    if total_relevant == 0:
        return 0.0
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("Inputs must be non-negative")
    return relevant_count / total_relevant


# The 20 Newsgroups dataset has ~500 docs per category
# If we retrieve top-5 and all 5 are in the right category:
#   Recall = 5 / ~500 = ~0.01 (1%)  ← very low - but expected at K=5
print("Recall@K examples:")
print(f"  5 relevant found, 500 total in corpus: {recall_at_k(5, 500):.4f}  (1%)")
print(f"  50 found, 500 total:                   {recall_at_k(50, 500):.4f}  (10%)")
print(f"  500 found, 500 total:                  {recall_at_k(500, 500):.4f}  (100% - only possible if K=N)")

---
## 4 - MAP@K: Mean Average Precision

MAP rewards retrievers that rank relevant documents **higher**, not just retrieve them.

For each query: compute precision at every rank where a relevant document appears, then average.
Then average that average across all queries.

$$AP@K = \frac{1}{|R|} \sum_{k=1}^{K} P@k \cdot rel(k)$$

where $rel(k)=1$ if document at rank $k$ is relevant, 0 otherwise.

In [ ]:
def average_precision_at_k(relevance_list: list) -> float:
    '''
    Average Precision for a single query.

    Args:
        relevance_list: Binary list - 1 if document at that rank is relevant, 0 if not.
                        Example: [1, 0, 1, 0, 0] means rank-1 and rank-3 were relevant.

    Returns:
        Average precision value.
    '''
    hits        = 0
    sum_prec    = 0.0
    for rank, rel in enumerate(relevance_list, start=1):
        if rel:
            hits     += 1
            sum_prec += hits / rank   # precision at this rank
    return sum_prec / max(hits, 1)


# Illustration
print("AP@K examples:")
print(f"  [1,0,0,0,0] - relevant at rank 1 only:  AP={average_precision_at_k([1,0,0,0,0]):.4f}")
print(f"  [0,0,0,0,1] - relevant at rank 5 only:  AP={average_precision_at_k([0,0,0,0,1]):.4f}")
print(f"  [1,1,0,0,0] - relevant at ranks 1,2:    AP={average_precision_at_k([1,1,0,0,0]):.4f}")
print(f"  [1,0,1,0,1] - relevant at ranks 1,3,5:  AP={average_precision_at_k([1,0,1,0,1]):.4f}")
print()
print("Putting a relevant doc at rank 1 gives much higher AP than putting it at rank 5.")

---
## 5 - MRR: Mean Reciprocal Rank

MRR measures how soon the **first** relevant document appears.
Useful when only the top result matters (e.g. a direct answer chatbot).

$$MRR = \frac{1}{|Q|} \sum_{q} \frac{1}{\text{rank of first relevant doc for } q}$$

In [ ]:
def reciprocal_rank(relevance_list: list) -> float:
    '''Reciprocal rank for a single query - 1/rank_of_first_relevant_doc.'''
    for rank, rel in enumerate(relevance_list, start=1):
        if rel:
            return 1.0 / rank
    return 0.0   # no relevant document found


print("RR examples:")
print(f"  [1,0,0,0,0] - first hit at rank 1: RR={reciprocal_rank([1,0,0,0,0]):.4f}")
print(f"  [0,1,0,0,0] - first hit at rank 2: RR={reciprocal_rank([0,1,0,0,0]):.4f}")
print(f"  [0,0,0,0,1] - first hit at rank 5: RR={reciprocal_rank([0,0,0,0,1]):.4f}")
print(f"  [0,0,0,0,0] - no hit:              RR={reciprocal_rank([0,0,0,0,0]):.4f}")

---
## 6 - Compute all metrics over real queries

In [ ]:
# Test queries with known correct categories
test_queries = [
    {"query": "advancements in space exploration technology",       "desired_category": "sci.space"},
    {"query": "real-time rendering techniques in computer graphics","desired_category": "comp.graphics"},
    {"query": "latest findings in cardiovascular medical research",  "desired_category": "sci.med"},
    {"query": "NHL playoffs and team performance statistics",        "desired_category": "rec.sport.hockey"},
    {"query": "impacts of cryptography in online security",         "desired_category": "sci.crypt"},
    {"query": "the role of electronics in modern computing devices","desired_category": "sci.electronics"},
    {"query": "motorcycles maintenance tips for enthusiasts",       "desired_category": "rec.motorcycles"},
    {"query": "high-performance baseball tactics for championships","desired_category": "rec.sport.baseball"},
    {"query": "historical influence of politics on society",        "desired_category": "talk.politics.misc"},
    {"query": "latest technology trends in Windows operating system","desired_category": "comp.os.ms-windows.misc"},
]


def evaluate_retriever(queries: list, top_k: int = 5) -> pd.DataFrame:
    '''
    Compute Precision@K, Recall@K, AP@K, and RR for each query.

    Returns a DataFrame with one row per query.
    '''
    results = []

    # Pre-compute total relevant docs per category (slow once, fast for all queries)
    cat_counts = {}
    for cat in newsgroups.target_names:
        cat_counts[cat] = sum(1 for c in newsgroups.target_names[df["category"]]
                              if c == cat)
    # Faster version
    for cat in newsgroups.target_names:
        cat_idx = newsgroups.target_names.tolist().index(cat)
        cat_counts[cat] = int((df["category"] == cat_idx).sum())

    for item in queries:
        query   = item["query"]
        cat_key = item["desired_category"]

        # Retrieve
        q_emb  = model.encode(query.strip()).astype(np.float32)
        q_emb /= np.linalg.norm(q_emb)
        scores = (E_norm @ q_emb).tolist()
        top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        # Build relevance list
        relevance = []
        for idx in top_idx:
            retrieved_cat = newsgroups.target_names[df.iloc[idx]["category"]]
            relevance.append(1 if retrieved_cat == cat_key else 0)

        relevant_in_top_k = sum(relevance)
        total_relevant    = cat_counts[cat_key]

        results.append({
            "query":    query[:50] + "...",
            "category": cat_key,
            f"P@{top_k}":  precision_at_k(relevant_in_top_k, top_k),
            f"R@{top_k}":  recall_at_k(relevant_in_top_k, total_relevant),
            f"AP@{top_k}": average_precision_at_k(relevance),
            "RR":          reciprocal_rank(relevance),
        })

    return pd.DataFrame(results)


# Run evaluation for K=5
results_k5 = evaluate_retriever(test_queries, top_k=5)
print("Retrieval metrics at K=5:")
print(results_k5.to_string(index=False))
print(f"\nMean P@5:  {results_k5['P@5'].mean():.4f}")
print(f"Mean R@5:  {results_k5['R@5'].mean():.4f}")
print(f"MAP@5:     {results_k5['AP@5'].mean():.4f}")
print(f"MRR:       {results_k5['RR'].mean():.4f}")

In [ ]:
# Compare K=5, K=20, K=50 - see the precision-recall tradeoff
import matplotlib.pyplot as plt

k_values = [5, 20, 50]
mean_precision = []
mean_recall    = []

for k in k_values:
    df_k = evaluate_retriever(test_queries, top_k=k)
    mean_precision.append(df_k[f"P@{k}"].mean())
    mean_recall.append(df_k[f"R@{k}"].mean())
    print(f"K={k:2d}:  Mean Precision={mean_precision[-1]:.4f}  Mean Recall={mean_recall[-1]:.4f}")

# Plot the tradeoff
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, mean_precision, "o-", color="#2980b9", label="Mean Precision@K")
ax.plot(k_values, mean_recall,    "s-", color="#e74c3c", label="Mean Recall@K")
ax.set_xlabel("K (number of documents retrieved)")
ax.set_ylabel("Score")
ax.set_title("Precision-Recall tradeoff as K increases")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

print()
print("As K increases:")
print("  • Recall increases - you find more of the relevant documents.")
print("  • Precision decreases - a larger fraction of retrieved docs are irrelevant.")
print("  • For RAG, K=5 to K=20 is typically optimal - high precision, manageable context size.")

---
## 7 - Exercise: implement MAP@K and MRR from scratch, then evaluate

In [ ]:
def mean_average_precision(queries_relevance: list) -> float:
    '''
    MAP@K across multiple queries.

    Args:
        queries_relevance: List of relevance lists, one per query.
                           Each is a binary list like [1, 0, 1, 0, 0].
    Returns:
        Mean Average Precision (float).
    '''
    # YOUR CODE HERE - use average_precision_at_k() for each query, then average
    pass


def mean_reciprocal_rank(queries_relevance: list) -> float:
    '''
    MRR across multiple queries.

    Args:
        queries_relevance: List of relevance lists, one per query.
    Returns:
        Mean Reciprocal Rank (float).
    '''
    # YOUR CODE HERE - use reciprocal_rank() for each query, then average
    pass


# Test on known inputs
test_relevance = [
    [1, 0, 0, 0, 0],   # query 1: relevant at rank 1
    [0, 0, 1, 0, 0],   # query 2: relevant at rank 3
    [0, 1, 0, 1, 0],   # query 3: relevant at ranks 2 and 4
]

map_score = mean_average_precision(test_relevance)
mrr_score = mean_reciprocal_rank(test_relevance)
print(f"MAP: {map_score:.4f}  (expected ~0.4444)")
print(f"MRR: {mrr_score:.4f}  (expected ~0.5556)")

assert map_score is not None, "Implement mean_average_precision"
assert mrr_score is not None, "Implement mean_reciprocal_rank"
print("\n✅ Exercise passed")

---
**Lab 05 complete.**

You can now measure retrieval quality with four metrics:
- **Precision@K** - are the retrieved docs relevant?
- **Recall@K** - did we find all the relevant docs?
- **MAP@K** - are relevant docs ranked near the top?
- **MRR** - how soon does the first relevant doc appear?

Use these metrics to compare retrievers and tune K, beta, and other hyperparameters.

**Assignment:** `assignments/assignment02_retrieval_pipeline.ipynb`